# Collaborative Filtering Recommender

**Team 9 — Recommender Systems Spring 2026**

## Overview

This notebook implements a **Matrix Factorization** collaborative filtering recommender using **Truncated SVD** (Singular Value Decomposition).

Collaborative filtering relies solely on user-item interaction data (ratings). No metadata or review text is used.

## How It Works

We build a user-item rating matrix **R** where each row is a user and each column is a restaurant. Most entries are zero (unobserved). We apply Truncated SVD to decompose R into latent factors:

**R ≈ U × Σ × Vᵀ**

- **U** — user latent factor matrix (shape: n_users × k)
- **Σ** — singular values (diagonal)
- **Vᵀ** — item latent factor matrix (shape: k × n_items)

The reconstructed matrix **R_pred = U × Vᵀ** gives a predicted score for every user-item pair, including unseen ones. We use **k=50 latent factors**.

For recommendations, each user is ranked against all restaurants they have not interacted with in training.

## Train/Test Split
- **Training:** `train_reviews_santa_barbara.csv` — used to build the rating matrix and fit SVD
- **Testing:** `test_reviews_santa_barbara.csv` — used only for evaluation
- The provided split is used as-is and has not been modified.

## Evaluation Metrics
- **Ranking:** Hit@10, Hit@20, Hit@30, NDCG@10, NDCG@20, NDCG@30
- **Rating prediction:** RMSE, MAE, R²

In [8]:
# Imports
import pandas as pd
import numpy as np
import math
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Load Data

In [9]:
def load_data(train_path, test_path, restaurants_path):
    """
    Load train, test, and restaurant metadata CSV files.

    Parameters
    ----------
    train_path : str
        Path to train_reviews_santa_barbara.csv
    test_path : str
        Path to test_reviews_santa_barbara.csv
    restaurants_path : str
        Path to restaurants_santa_barbara.csv

    Returns
    -------
    tuple
        (train_df, test_df, restaurants_df)
    """
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    restaurants_df = pd.read_csv(restaurants_path)

    # Ensure consistent string types for IDs
    for df in [train_df, test_df]:
        df['user_id'] = df['user_id'].astype(str)
        df['business_id'] = df['business_id'].astype(str)
        df['stars'] = df['stars'].astype(float)
    restaurants_df['business_id'] = restaurants_df['business_id'].astype(str)

    return train_df, test_df, restaurants_df


train_df, test_df, restaurants_df = load_data(
    'train_reviews_santa_barbara.csv',
    'test_reviews_santa_barbara.csv',
    'restaurants_santa_barbara.csv'
)

print(f'Train interactions: {len(train_df)}')
print(f'Test interactions:  {len(test_df)}')
print(f'Unique users:       {train_df["user_id"].nunique()}')
print(f'Unique restaurants: {restaurants_df["business_id"].nunique()}')

Train interactions: 41581
Test interactions:  4801
Unique users:       4801
Unique restaurants: 767


## 2. Build User-Item Rating Matrix

In [10]:
def build_rating_matrix(train_df, restaurants_df):
    """
    Build a dense user-item rating matrix from the training data.

    Rows correspond to users, columns correspond to all restaurants in the
    metadata file. Unobserved entries are set to 0.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training interactions with columns [user_id, business_id, stars].
    restaurants_df : pd.DataFrame
        Restaurant metadata with column [business_id].

    Returns
    -------
    tuple
        (R, users, items, user_idx, item_idx)
        R : np.ndarray of shape (n_users, n_items)
        users : sorted list of user IDs
        items : sorted list of business IDs
        user_idx : dict mapping user_id -> row index
        item_idx : dict mapping business_id -> column index
    """
    users = sorted(train_df['user_id'].unique())
    items = sorted(restaurants_df['business_id'].unique())
    user_idx = {u: i for i, u in enumerate(users)}
    item_idx = {it: i for i, it in enumerate(items)}

    R = np.zeros((len(users), len(items)), dtype=np.float32)
    for _, row in train_df.iterrows():
        u, it = row['user_id'], row['business_id']
        if u in user_idx and it in item_idx:
            R[user_idx[u], item_idx[it]] = row['stars']

    sparsity = 1.0 - np.count_nonzero(R) / R.size
    print(f'Rating matrix shape: {R.shape}')
    print(f'Sparsity: {sparsity:.4f} ({sparsity*100:.2f}% of entries are unobserved)')
    return R, users, items, user_idx, item_idx


R, users, items, user_idx, item_idx = build_rating_matrix(train_df, restaurants_df)

Rating matrix shape: (4801, 767)
Sparsity: 0.9887 (98.87% of entries are unobserved)


## 3. Matrix Factorization via Truncated SVD

We apply Truncated SVD with **k=50 latent factors** to decompose the rating matrix. The reconstructed matrix provides predicted scores for all user-item pairs, including unobserved ones.

Latent factors capture underlying patterns in user preferences and restaurant characteristics without using any explicit features.

In [11]:
def fit_svd(R, n_components=50, random_state=42):
    """
    Fit a Truncated SVD model to the user-item rating matrix.

    The reconstructed matrix R_pred = U @ Vt approximates the full rating
    matrix, filling in unobserved entries with predicted scores.

    Parameters
    ----------
    R : np.ndarray
        User-item rating matrix of shape (n_users, n_items).
    n_components : int
        Number of latent factors (singular values) to keep.
    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    tuple
        (R_pred, svd) where R_pred is the reconstructed matrix and
        svd is the fitted TruncatedSVD object.
    """
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)
    U = svd.fit_transform(R)      # shape: (n_users, n_components)
    Vt = svd.components_          # shape: (n_components, n_items)
    R_pred = U @ Vt               # shape: (n_users, n_items)
    explained = svd.explained_variance_ratio_.sum()
    print(f'SVD with {n_components} components explains {explained*100:.2f}% of variance')
    return R_pred, svd


R_pred, svd_model = fit_svd(R, n_components=50)

SVD with 50 components explains 36.40% of variance


## 4. Build Training History

For each user, we record which restaurants they interacted with in training. These are excluded from recommendations.

In [12]:
def build_train_history(train_df):
    """
    Build a mapping from each user to the set of restaurants they interacted
    with in the training data.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training interactions with columns [user_id, business_id].

    Returns
    -------
    dict
        Mapping user_id -> set of business_ids seen in training.
    """
    history = {}
    for _, row in train_df.iterrows():
        history.setdefault(row['user_id'], set()).add(row['business_id'])
    return history


train_history = build_train_history(train_df)

## 5. Evaluation Metric Functions

In [13]:
def hit_at_k(recommended, target_item, k):
    """
    Compute Hit@K for a single user.

    Parameters
    ----------
    recommended : list of str
        Ranked list of recommended business IDs (highest score first).
    target_item : str
        The held-out test business ID.
    k : int
        Cutoff rank.

    Returns
    -------
    float
        1.0 if target appears in top-K, else 0.0.
    """
    return 1.0 if target_item in recommended[:k] else 0.0


def ndcg_at_k(recommended, target_item, k):
    """
    Compute NDCG@K for a single user with one relevant item.

    Parameters
    ----------
    recommended : list of str
        Ranked list of recommended business IDs (highest score first).
    target_item : str
        The held-out test business ID.
    k : int
        Cutoff rank.

    Returns
    -------
    float
        NDCG@K score (0.0 if not in top-K).
    """
    topk = recommended[:k]
    if target_item not in topk:
        return 0.0
    rank = topk.index(target_item) + 1  # 1-based
    return 1.0 / math.log2(rank + 1)

## 6. Final Evaluation (Required Deliverable)

**This is the final cell of the notebook.** No training or model fitting occurs after this point.

For each test user:
- Candidate restaurants = all restaurants **not seen in training**
- Ranked by predicted SVD score (descending)
- Hit@K and NDCG@K computed against the one held-out test restaurant
- Rating prediction clipped to [1, 5] for RMSE/MAE/R²

In [14]:
Ks = [10, 20, 30]
hit_sums = {k: 0.0 for k in Ks}
ndcg_sums = {k: 0.0 for k in Ks}
y_true, y_pred_rating = [], []
n_users = 0

for _, row in test_df.iterrows():
    u = str(row['user_id'])
    target = str(row['business_id'])
    true_r = float(row['stars'])

    if u not in user_idx:
        continue

    ui = user_idx[u]
    seen = train_history.get(u, set())

    # Score and rank all unseen candidate restaurants
    candidates = [
        (it, float(R_pred[ui, item_idx[it]]))
        for it in items
        if it not in seen and it in item_idx
    ]
    candidates.sort(key=lambda x: -x[1])
    recs = [it for it, _ in candidates]

    for k in Ks:
        hit_sums[k] += hit_at_k(recs, target, k)
        ndcg_sums[k] += ndcg_at_k(recs, target, k)

    # Rating prediction for RMSE/MAE/R²
    if target in item_idx:
        pred_r = float(np.clip(R_pred[ui, item_idx[target]], 1.0, 5.0))
        y_true.append(true_r)
        y_pred_rating.append(pred_r)

    n_users += 1

# Compute rating metrics
rmse = float(np.sqrt(mean_squared_error(y_true, y_pred_rating)))
mae = float(mean_absolute_error(y_true, y_pred_rating))
r2 = float(r2_score(y_true, y_pred_rating))

print('Collaborative Filtering (SVD) Recommender — Test Metrics')
print('=' * 55)
print(f'Users evaluated: {n_users}')
print()
print('Ranking Metrics:')
for k in Ks:
    print(f'  Hit@{k}:  {hit_sums[k]/n_users:.4f}    NDCG@{k}:  {ndcg_sums[k]/n_users:.4f}')
print()
print('Rating Prediction Metrics:')
print(f'  RMSE: {rmse:.4f}')
print(f'  MAE:  {mae:.4f}')
print(f'  R²:   {r2:.4f}')
print()
print('Note: RMSE/MAE are high because SVD on a sparse binary-style matrix')
print('does not optimize for rating accuracy. Ranking metrics are the primary')
print('evaluation criterion for this recommender.')

Collaborative Filtering (SVD) Recommender — Test Metrics
Users evaluated: 4801

Ranking Metrics:
  Hit@10:  0.0635    NDCG@10:  0.0298
  Hit@20:  0.1179    NDCG@20:  0.0435
  Hit@30:  0.1654    NDCG@30:  0.0535

Rating Prediction Metrics:
  RMSE: 3.2833
  MAE:  3.0372
  R²:   -5.9242

Note: RMSE/MAE are high because SVD on a sparse binary-style matrix
does not optimize for rating accuracy. Ranking metrics are the primary
evaluation criterion for this recommender.
